<a href="https://colab.research.google.com/github/Shrushtid13/Simple-RAG-implementation/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [101]:
from google.colab import userdata
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
GROQ_API_KEY = userdata.get('GROQ_API_KEY')


In [102]:
!pip install -q langchain==1.0.5 langchain-groq==1.0.0 langchain-google-genai==3.0.1 langchain-community==0.4.1 chromadb==1.3.4 pypdf


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-chroma 1.1.0 requires chromadb<2.0.0,>=1.3.5, but you have chromadb 1.3.4 which is incompatible.


In [103]:
!pip show langchain langchain-groq langchain-google-genai langchain-community chromadb pypdf


Name: langchain
Version: 1.0.5
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
---
Name: langchain-groq
Version: 1.0.0
Summary: An integration package connecting Groq and LangChain
Home-page: https://docs.langchain.com/oss/python/integrations/providers/groq
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: groq, langchain-core
Required-by: 
---
Name: langchain-google-genai
Version: 3.0.1
Summary: An integration package connecting Google's genai package and LangChain
Home-page: 
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: filetype, google-ai-generativelanguage, langchain-core, pydantic
Required-by: 
---
Name: langchain-community
Version: 0.4.1
Summary: Community contributed LangChain integra

In [104]:
from langchain_community.document_loaders import PyPDFLoader
file_path="/content/scholarship_info.pdf"
loader= PyPDFLoader(file_path)
doc=loader.load()

In [105]:
doc

[Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2025-08-07T19:16:29+05:30', 'author': 'Preethesh Poojary', 'moddate': '2025-08-07T19:16:29+05:30', 'source': '/content/scholarship_info.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Title: Scholarship Information 2025 \n \n1. Eligibility: \n- Open to students in India pursuing undergraduate degrees. \n- Annual family income must be below ₹6,00,000. \n- Minimum 60% marks in the last qualifying exam. \n \n2. Documents Required: \n- Income certificate \n- Aadhaar card \n- Bank passbook \n- Marksheet \n \n3. Deadline: October 15, 2025 \n \n4. Benefits: \n- ₹10,000 per semester for tuition \n- Book allowance of ₹3,000 per year \n \n5. How to Apply: \nVisit https://scholarships.gov.in and register under the NSP portal.')]

In [106]:
from langchain_text_splitters  import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap  = 200
)
splits = splitter.split_documents(doc)
print(f"Splits into {len(splits)}")

Splits into 1


In [107]:
!pip install google-genai

In [108]:
!pip install -qU "langchain-chroma>=0.1.2"

In [111]:
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2-preview",
    google_api_key = GEMINI_API_KEY
)

vectore_store = Chroma(
    embedding_function=embeddings,
    persist_directory="r_db",
    collection_name="scholarship_info"
)

vectore_store.add_documents(splits)

['03de02d3-55fe-4f54-95bd-e4069c479d36']

In [112]:
retriever = vectore_store.as_retriever(search_kwargs={"k": 2})

In [113]:
from langchain_groq import ChatGroq

llm = ChatGroq(model = "openai/gpt-oss-120b",
               api_key = GROQ_API_KEY,
               temperature = 0.3,
               max_tokens= 200)

In [114]:
from langchain_classic.chains import RetrievalQA
rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectore_store.as_retriever()
)

In [115]:
query = "who is eligible for the schloarship?"
response = rag_chain.invoke({"query" : query})
print(response["result"])

**Eligibility for the scholarship**

- **Location & Level:** Students **in India** who are pursuing an **undergraduate degree**.  
- **Family Income:** The total annual family income must be **below ₹6,00,000**.  
- **Academic Performance:** The student must have scored **at least 60 %** in the last qualifying examination.  

If a student meets all three of these conditions, they are eligible to apply for the scholarship.


In [116]:
query = "what are the benefits of the schloarship?"
response = rag_chain.invoke({"query" : query})
print(response["result"])

The scholarship provides the following benefits:

- **Tuition support:** ₹10,000 per semester  
- **Book allowance:** ₹3,000 per year  

These amounts are intended to help cover your tuition fees and the cost of textbooks.
